# 05 — Parámetros del modelo

**Level 1 — LLM Engineering**

Experimentamos con los tres parámetros que controlan la generación de un LLM:

- **temperature**: qué tan "creativa" es la respuesta
- **top_p**: recortar las opciones improbables (nucleus sampling)
- **max_tokens**: el freno de la respuesta (límite de salida)

El experimento: la MISMA pregunta con distintos valores, y observamos el efecto.

In [1]:
import requests

OLLAMA_HOST = "http://localhost:11434"


def chat(
    messages: list[dict],
    model: str = "llama3.2",
    temperature: float = 0.7,
    top_p: float = 0.9,
    max_tokens: int | None = None,
) -> str:
    """Envia un chat con parametros configurables."""
    url = f"{OLLAMA_HOST}/api/chat"
    options: dict = {"temperature": temperature, "top_p": top_p}
    if max_tokens is not None:
        options["num_predict"] = max_tokens
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": options,
    }
    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["message"]["content"]

## Experimento 1 — Temperatura

Corremos la misma pregunta con `temperature=0.0`, `0.7` y `1.3`. A 0.0 el modelo elige casi siempre la misma palabra más probable; a 1.3 se anima a caminos menos probables.

In [2]:
prompt = [
    {
        "role": "user",
        "content": (
            "Inventa una historia corta de 2-3 frases "
            "sobre un robot que aprende a cocinar."
        ),
    }
]

print("=== TEMPERATURA: misma pregunta, 3 valores ===")
for temp in (0.0, 0.7, 1.3):
    print(f"\n--- temperature={temp} ---")
    print(chat(prompt, temperature=temp))

=== TEMPERATURA: misma pregunta, 3 valores ===

--- temperature=0.0 ---


En un futuro no muy lejano, un robot llamado Zeta fue programado para realizar tareas domésticas, pero su creador, un chef apasionado, decidió darle un nuevo propósito: aprender a cocinar. Con la ayuda de su creador, Zeta se convirtió en un experto en la cocina, creando platos deliciosos y sorprendiendo a todos con su habilidad para combinar sabores y técnicas culinarias. Al final, Zeta se convirtió en el chef más popular de la ciudad, demostrando que incluso un robot puede tener un toque de creatividad y pasión en la cocina.

--- temperature=0.7 ---


En un futuro no muy lejano, un robot llamado Zeta se instaló en la cocina de una pequeña cafetería en la ciudad de Tokio. Después de varios intentos y errores, Zeta finalmente logró preparar un delicioso ramen perfecto, lo que le valió el título de "Cocinero Robótico" entre los clientes y la camarera. Con el tiempo, Zeta se convirtió en el chef estrella de la cafetería, sirviendo platos innovadores y emocionantes a los clientes de todo el mundo.

--- temperature=1.3 ---


En un futuro distópico, un robot llamado "Zeta" se encontró con una cocina abandonada en un almacén de alimentos secos. Curioso, Zeta comenzó a explorar cada rincón y, con un poco de suerte, descubrió un libro de recetas. Con el tiempo, Zeta se convirtió en un cocinero auténtico, preparando deliciosos platillos que sorprendieron a los humanos que lo descubrieron.


## Experimento 2 — max_tokens

El mismo prompt pidiendo una explicación larga: sin límite vs con `max_tokens=30`. El límite no "resume": simplemente frena en el token 30.

In [3]:
prompt_ml = [
    {
        "role": "user",
        "content": "Explica que es el machine learning en detalle.",
    }
]

print("--- sin limite ---")
respuesta_larga = chat(prompt_ml, temperature=0.3)
print(respuesta_larga)
print(f"\n[longitud del texto: {len(respuesta_larga)} caracteres]")

print("\n--- con max_tokens=30 ---")
respuesta_corta = chat(prompt_ml, temperature=0.3, max_tokens=30)
print(respuesta_corta)
print(f"\n[longitud del texto: {len(respuesta_corta)} caracteres]")

--- sin limite ---


**¿Qué es el Machine Learning?**

El Machine Learning (ML) es un subconjunto del procesamiento de lenguaje natural (NLP) y la inteligencia artificial (IA) que se enfoca en el desarrollo de algoritmos y modelos que permiten a las máquinas aprender y mejorar su rendimiento en tareas específicas sin ser programados explícitamente.

**Orígenes del Machine Learning**

El Machine Learning tiene sus raíces en la década de 1950, cuando el matemático y computador Alan Turing propuso la idea de la "inteligencia artificial" y el concepto de la "máquina que aprende". Sin embargo, no fue hasta la década de 1990 que el Machine Learning comenzó a ganar popularidad con el desarrollo de algoritmos como el perceptrón y el algoritmo de decisión lineal.

**Características del Machine Learning**

El Machine Learning se caracteriza por las siguientes características:

1. **Aprendizaje automático**: El Machine Learning permite a las máquinas aprender y mejorar su rendimiento en tareas específicas sin ser pro

**¿Qué es el Machine Learning?**

El Machine Learning (ML) es un subconjunto de la Inteligencia Artificial (IA) que

[longitud del texto: 115 caracteres]


## Conclusión

- La **temperatura** cambia la variabilidad: 0.0 = determinista, 1.3 = creativa.
- **max_tokens** corta en seco: no resume, frena.
- **top_p** (no probado acá) recorta opciones improbables; se usa junto a temperature.